<a href="https://colab.research.google.com/github/Jefrina-V/VAC-Artificial-Intelligence-and-Data-Science-/blob/main/Day4_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install numpy tensorflow

In [ ]:
import pickle
import numpy as np
import tensorflow

In [ ]:
from keras.models import Sequential
from keras.layers import Dense

In [ ]:
from pathlib import Path
import pickle
# Please upload 'train_qa.txt' to your Colab environment.
# For example, if you upload it to the root of your Colab session,
# it will typically be accessible at '/content/train_qa.txt'.
path = Path("/content/train_qa.txt")
with open(path, "rb") as fp:
    train_data = pickle.load(fp)


In [ ]:
from pathlib import Path
import pickle
# Please upload 'test_qa.txt' to your Colab environment.
# For example, if you upload it to the root of your Colab session,
# it will typically be accessible at '/content/test_qa.txt'.
path = Path("/content/test_qa.txt")
with open(path, "rb") as fp:
    test_data = pickle.load(fp)


In [34]:
' '.join(train_data[0][0])

'Mary moved to the bathroom . Sandra journeyed to the bedroom .'

In [35]:


type(test_data)

list

In [36]:
' '.join(test_data[0][0])



'Mary got the milk there . John moved to the bedroom .'

In [37]:
' '.join(test_data[0][1])



'Is John in the kitchen ?'

In [38]:
' '.join(test_data[1][0])



'Mary got the milk there . John moved to the bedroom . Mary discarded the milk . John went to the garden .'

In [39]:
' '.join(test_data[1][1])



'Is John in the kitchen ?'

In [40]:
' '.join(test_data[0][2])



'n o'

In [41]:
' '.join(test_data[1][2])



'n o'

In [42]:
' '.join(test_data[0][0][1])

'g o t'

In [43]:
len(train_data)

10000

In [44]:
vocab=set()

In [45]:
all_data=test_data+train_data


In [46]:
for story,question,answer in all_data:
    vocab=vocab.union(set(story))
    vocab=vocab.union(set(question))


In [48]:
vocab.add('no')
vocab.add('yes')
vocab

{'.',
 '?',
 'Daniel',
 'Is',
 'John',
 'Mary',
 'Sandra',
 'apple',
 'back',
 'bathroom',
 'bedroom',
 'discarded',
 'down',
 'dropped',
 'football',
 'garden',
 'got',
 'grabbed',
 'hallway',
 'in',
 'journeyed',
 'kitchen',
 'left',
 'milk',
 'moved',
 'no',
 'office',
 'picked',
 'put',
 'the',
 'there',
 'to',
 'took',
 'travelled',
 'up',
 'went',
 'yes'}

In [49]:
vocab_len=len(vocab)+1

In [50]:
max_story_len=max([len(data[0]) for data in all_data])

max_story_len

max_question_len=max([len(data[1]) for data in all_data])

max_question_len

6

In [51]:
vocab

{'.',
 '?',
 'Daniel',
 'Is',
 'John',
 'Mary',
 'Sandra',
 'apple',
 'back',
 'bathroom',
 'bedroom',
 'discarded',
 'down',
 'dropped',
 'football',
 'garden',
 'got',
 'grabbed',
 'hallway',
 'in',
 'journeyed',
 'kitchen',
 'left',
 'milk',
 'moved',
 'no',
 'office',
 'picked',
 'put',
 'the',
 'there',
 'to',
 'took',
 'travelled',
 'up',
 'went',
 'yes'}

In [52]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer=Tokenizer()

vocab_size=len(tokenizer.word_index)+1
print("Vocab size:",vocab_size)


Vocab size: 1


In [53]:
import keras

In [55]:
!pip install keras --user

from keras.preprocessing.sequence import pad_sequences

text=['Mary moved to the bathroom,Sandra journeyed to the bedroom.']

tokenizer=Tokenizer(num_words=18)

tokenizer.fit_on_texts(text)
sequences=tokenizer.texts_to_sequences(text)


In [56]:
tokenizer=Tokenizer(filters=[])
tokenizer.fit_on_texts(vocab)

tokenizer.word_index


{'to': 1,
 'got': 2,
 'bathroom': 3,
 'bedroom': 4,
 'mary': 5,
 'sandra': 6,
 'football': 7,
 'hallway': 8,
 'moved': 9,
 'daniel': 10,
 'is': 11,
 'apple': 12,
 'went': 13,
 '.': 14,
 'office': 15,
 'kitchen': 16,
 '?': 17,
 'grabbed': 18,
 'garden': 19,
 'travelled': 20,
 'back': 21,
 'the': 22,
 'in': 23,
 'dropped': 24,
 'yes': 25,
 'left': 26,
 'milk': 27,
 'took': 28,
 'there': 29,
 'discarded': 30,
 'up': 31,
 'picked': 32,
 'down': 33,
 'put': 34,
 'no': 35,
 'journeyed': 36,
 'john': 37}

In [57]:
train_story_text=[]
train_question_text=[]
train_answers=[]

In [58]:
for story,question,answer in train_data:
    train_story_text.append(story)
    train_question_text.append(question)

In [59]:
train_story_seq=tokenizer.texts_to_sequences(train_story_text)

In [60]:
len(train_story_text)

len(train_story_seq)

10000

In [63]:
def vectorize_stories(data, word_index=tokenizer.word_index, max_story_len=max_story_len, max_question_len=max_question_len):
    # Initialize lists to store all vectorized stories, questions, and answers
    all_stories = []
    all_questions = []
    all_answers = []

    # Ensure the '<unk>' token exists in the word_index
    if '<unk>' not in word_index:
        word_index['<unk>'] = len(word_index) + 1

    for story, query, answer in data:
        # Vectorize the current story, using '<unk>' for unknown words
        current_story_vec = [word_index.get(word.lower(), word_index['<unk>']) for word in story]
        # Vectorize the current question
        current_question_vec = [word_index.get(word.lower(), word_index['<unk>']) for word in query]

        # Vectorize the current answer (one-hot encoding)
        current_answer_vec = np.zeros(len(word_index) + 1)
        # Ensure the answer is in word_index before accessing it
        if answer in word_index:
            current_answer_vec[word_index[answer]] = 1
        else:
            # Fallback to <unk> for answers if not found (though for this dataset, 'yes'/'no' are in vocab)
            current_answer_vec[word_index['<unk>']] = 1

        # Append the vectorized items to the collecting lists
        all_stories.append(current_story_vec)
        all_questions.append(current_question_vec)
        all_answers.append(current_answer_vec)

    # Pad sequences to ensure uniform length for stories and questions
    padded_stories = pad_sequences(all_stories, maxlen=max_story_len)
    padded_questions = pad_sequences(all_questions, maxlen=max_question_len)

    # Convert answers list to a numpy array
    np_answers = np.array(all_answers)

    return (padded_stories, padded_questions, np_answers)

In [64]:
inputs_train,queries_train,answers_train=vectorize_stories(train_data)

In [66]:
inputs_train.shape

answers_train.shape

(10000, 40)

In [65]:
inputs_test,queries_test,answers_test=vectorize_stories(test_data)

In [67]:
inputs_test

array([[ 0,  0,  0, ..., 22,  4, 14],
       [ 0,  0,  0, ..., 22, 19, 14],
       [ 0,  0,  0, ..., 22, 19, 14],
       ...,
       [ 0,  0,  0, ..., 22, 12, 14],
       [ 0,  0,  0, ..., 22, 19, 14],
       [ 0,  0,  0, ..., 12, 29, 14]], dtype=int32)